In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
SRC_DIR = PROJECT_ROOT / "src"

TEST_FEATURE_PATH = PROCESSED_DATA_DIR / "ml_test_features.csv"
RULE_CONFIG_PATH = MODELS_DIR / "rule_engine_config.json"
ML_MODEL_PATH = MODELS_DIR / "xgboost_fraud_detector.json"

if not TEST_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Test feature file not found:\n{TEST_FEATURE_PATH}"
    )

if not RULE_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Rule configuration not found:\n{RULE_CONFIG_PATH}"
    )

if not ML_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"ML model not found:\n{ML_MODEL_PATH}"
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rule_engine import evaluate_rules

df = pd.read_csv(TEST_FEATURE_PATH)

with open(RULE_CONFIG_PATH, "r") as file:
    rule_config = json.load(file)

ML_FEATURES = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

REQUIRED_COLUMNS = [
    "transaction_id",
    "user_id",
    *ML_FEATURES,
    "is_fraud"
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

DECISION_CONFIG = {
    "ml_weight": 0.5,
    "rule_weight": 0.5,
    "allow_max_risk": 0.25,
    "otp_max_risk": 0.50,
    "review_max_risk": 0.75
}

print("========== PHASE 10 INITIALIZATION ==========")
print("Test rows:", len(df))
print("ML weight:", DECISION_CONFIG["ml_weight"])
print("Rule weight:", DECISION_CONFIG["rule_weight"])
print("Allow maximum risk:", DECISION_CONFIG["allow_max_risk"])
print("OTP maximum risk:", DECISION_CONFIG["otp_max_risk"])
print("Review maximum risk:", DECISION_CONFIG["review_max_risk"])
print("Rule engine loaded:", callable(evaluate_rules))
print("=============================================")

========== PHASE 10 INITIALIZATION ==========
Test rows: 555719
ML weight: 0.5
Rule weight: 0.5
Allow maximum risk: 0.25
OTP maximum risk: 0.5
Review maximum risk: 0.75
Rule engine loaded: True


In [2]:
if not np.isclose(
    DECISION_CONFIG["ml_weight"]
    + DECISION_CONFIG["rule_weight"],
    1.0
):
    raise ValueError(
        "ML and rule weights must sum to 1.0."
    )

if not (
    0
    <= DECISION_CONFIG["allow_max_risk"]
    < DECISION_CONFIG["otp_max_risk"]
    < DECISION_CONFIG["review_max_risk"]
    <= 1
):
    raise ValueError(
        "Decision thresholds must be ordered between 0 and 1."
    )

print("========== DECISION CONFIG VALIDATION ==========")
print(
    "Weights sum to 1:",
    np.isclose(
        DECISION_CONFIG["ml_weight"]
        + DECISION_CONFIG["rule_weight"],
        1.0
    )
)

print(
    "Threshold ordering valid:",
    (
        0
        <= DECISION_CONFIG["allow_max_risk"]
        < DECISION_CONFIG["otp_max_risk"]
        < DECISION_CONFIG["review_max_risk"]
        <= 1
    )
)

print("===============================================")

========== DECISION CONFIG VALIDATION ==========
Weights sum to 1: True
Threshold ordering valid: True


In [3]:
def calculate_risk_score(
    ml_fraud_score,
    rule_score,
    config
):
    ml_fraud_score = float(
        ml_fraud_score
    )

    rule_score = float(
        rule_score
    )

    if not 0 <= ml_fraud_score <= 1:
        raise ValueError(
            "ML fraud score must be between 0 and 1."
        )

    if not 0 <= rule_score <= 1:
        raise ValueError(
            "Rule score must be between 0 and 1."
        )

    risk_score = (
        config["ml_weight"]
        * ml_fraud_score
        +
        config["rule_weight"]
        * rule_score
    )

    return float(
        np.clip(
            risk_score,
            0.0,
            1.0
        )
    )


print("========== RISK SCORE FUNCTION ==========")
print(
    "Risk-score function created:",
    callable(calculate_risk_score)
)
print("=========================================")

========== RISK SCORE FUNCTION ==========
Risk-score function created: True


In [4]:
def classify_risk(
    risk_score,
    config
):
    risk_score = float(
        risk_score
    )

    if not 0 <= risk_score <= 1:
        raise ValueError(
            "Risk score must be between 0 and 1."
        )

    if risk_score <= config["allow_max_risk"]:
        return "allow"

    if risk_score <= config["otp_max_risk"]:
        return "otp"

    if risk_score <= config["review_max_risk"]:
        return "review"

    return "block"


print("========== DECISION CLASSIFIER ==========")
print(
    "Decision classifier created:",
    callable(classify_risk)
)
print("=========================================")

========== DECISION CLASSIFIER ==========
Decision classifier created: True


In [5]:
def generate_decision_reason(
    ml_fraud_score,
    rule_flags,
    risk_score,
    decision
):
    ml_fraud_score = float(
        ml_fraud_score
    )

    risk_score = float(
        risk_score
    )

    if not isinstance(
        rule_flags,
        list
    ):
        raise TypeError(
            "rule_flags must be a list."
        )

    parts = []

    if rule_flags:
        rule_text = ", ".join(
            rule_flags
        )
        parts.append(
            f"Triggered rules: {rule_text}"
        )
    else:
        parts.append(
            "No deterministic rules triggered"
        )

    parts.append(
        f"ML fraud score: {ml_fraud_score:.4f}"
    )

    parts.append(
        f"Combined risk score: {risk_score:.4f}"
    )

    parts.append(
        f"Decision: {decision.upper()}"
    )

    return (
        ". ".join(parts)
        + "."
    )


print("========== REASON GENERATOR ==========")
print(
    "Reason generator created:",
    callable(generate_decision_reason)
)
print("======================================")

========== REASON GENERATOR ==========
Reason generator created: True


In [6]:
def make_decision(
    ml_fraud_score,
    rule_result,
    config
):
    if not isinstance(
        rule_result,
        dict
    ):
        raise TypeError(
            "rule_result must be a dictionary."
        )

    if "rule_flags" not in rule_result:
        raise ValueError(
            "rule_result is missing rule_flags."
        )

    if "rule_score" not in rule_result:
        raise ValueError(
            "rule_result is missing rule_score."
        )

    rule_flags = rule_result[
        "rule_flags"
    ]

    rule_score = float(
        rule_result[
            "rule_score"
        ]
    )

    risk_score = calculate_risk_score(
        ml_fraud_score,
        rule_score,
        config
    )

    decision = classify_risk(
        risk_score,
        config
    )

    human_readable_reason = (
        generate_decision_reason(
            ml_fraud_score,
            rule_flags,
            risk_score,
            decision
        )
    )

    return {
        "risk_score": risk_score,
        "rule_flags": rule_flags,
        "ml_fraud_score": float(
            ml_fraud_score
        ),
        "decision": decision,
        "human_readable_reason": human_readable_reason
    }


print("========== COMPLETE DECISION ENGINE ==========")
print(
    "Decision function created:",
    callable(make_decision)
)
print("===============================================")

========== COMPLETE DECISION ENGINE ==========
Decision function created: True


In [20]:
decision_test_cases = {
    "allow": {
        "ml_score": 0.10,
        "rule_result": {
            "rule_flags": [],
            "rule_score": 0.0
        }
    },
    "otp": {
        "ml_score": 0.60,
        "rule_result": {
            "rule_flags": [],
            "rule_score": 0.0
        }
    },
    "review": {
        "ml_score": 0.70,
        "rule_result": {
            "rule_flags": [
                "HIGH_AMOUNT"
            ],
            "rule_score": 0.50
        }
    },
    "block": {
        "ml_score": 0.90,
        "rule_result": {
            "rule_flags": [
                "HIGH_AMOUNT",
                "HIGH_VELOCITY"
            ],
            "rule_score": 1.0
        }
    }
}

decision_test_results = {}

for expected_decision, case in decision_test_cases.items():
    result = make_decision(
        case["ml_score"],
        case["rule_result"],
        DECISION_CONFIG
    )

    decision_test_results[
        expected_decision
    ] = result

print("========== FOUR DECISION TESTS ==========")

for expected_decision, result in decision_test_results.items():
    print()
    print("Expected:", expected_decision)
    print("Actual:", result["decision"])
    print("Risk score:", result["risk_score"])
    print("Reason:", result["human_readable_reason"])

print("=========================================")

========== FOUR DECISION TESTS ==========

Expected: allow
Actual: allow
Risk score: 0.05
Reason: No deterministic rules triggered. ML fraud score: 0.1000. Combined risk score: 0.0500. Decision: ALLOW.

Expected: otp
Actual: otp
Risk score: 0.3
Reason: No deterministic rules triggered. ML fraud score: 0.6000. Combined risk score: 0.3000. Decision: OTP.

Expected: review
Actual: review
Risk score: 0.6
Reason: Triggered rules: HIGH_AMOUNT. ML fraud score: 0.7000. Combined risk score: 0.6000. Decision: REVIEW.

Expected: block
Actual: block
Risk score: 0.95
Reason: Triggered rules: HIGH_AMOUNT, HIGH_VELOCITY. ML fraud score: 0.9000. Combined risk score: 0.9500. Decision: BLOCK.


In [21]:
decision_tests_passed = {}

for expected_decision, result in decision_test_results.items():
    decision_tests_passed[
        expected_decision
    ] = (
        result["decision"]
        == expected_decision
    )

print("========== DECISION VALIDATION ==========")

for decision, status in decision_tests_passed.items():
    print(
        f"{decision.upper()}:",
        status
    )

print()

print(
    "All four decision tests passed:",
    all(
        decision_tests_passed.values()
    )
)

print("=========================================")

========== DECISION VALIDATION ==========
ALLOW: True
OTP: True
REVIEW: True
BLOCK: True

All four decision tests passed: True


In [9]:
integration_transaction = {
    "amount": 2000.0,
    "amount_vs_avg_ratio": 8.0,
    "txn_count_last_5min": 10,
    "time_since_last_txn_sec": 120.0,
    "distance_from_last_location_km": 1200.0,
    "merchant_category_is_new_for_user": 1
}

integration_rule_result = evaluate_rules(
    integration_transaction
)

integration_decision = make_decision(
    ml_fraud_score=0.80,
    rule_result=integration_rule_result,
    config=DECISION_CONFIG
)

print("========== RULE + DECISION INTEGRATION ==========")
print(
    "Rule result:",
    integration_rule_result
)

print()

print(
    "Decision result:",
    integration_decision
)

print("==================================================")

========== RULE + DECISION INTEGRATION ==========
Rule result: {'rule_flags': ['HIGH_AMOUNT', 'HIGH_VELOCITY', 'IMPOSSIBLE_TRAVEL', 'NEW_MERCHANT_CATEGORY'], 'rule_score': 1.0, 'rules_triggered': 4}

Decision result: {'risk_score': 0.9, 'rule_flags': ['HIGH_AMOUNT', 'HIGH_VELOCITY', 'IMPOSSIBLE_TRAVEL', 'NEW_MERCHANT_CATEGORY'], 'ml_fraud_score': 0.8, 'decision': 'block', 'human_readable_reason': 'Triggered rules: HIGH_AMOUNT, HIGH_VELOCITY, IMPOSSIBLE_TRAVEL, NEW_MERCHANT_CATEGORY. ML fraud score: 0.8000. Combined risk score: 0.9000. Decision: BLOCK.'}


In [10]:
required_decision_fields = [
    "risk_score",
    "rule_flags",
    "ml_fraud_score",
    "decision",
    "human_readable_reason"
]

missing_decision_fields = [
    field
    for field in required_decision_fields
    if field not in integration_decision
]

risk_score_valid = (
    0
    <= integration_decision["risk_score"]
    <= 1
)

ml_score_valid = (
    0
    <= integration_decision["ml_fraud_score"]
    <= 1
)

decision_valid = (
    integration_decision["decision"]
    in [
        "allow",
        "otp",
        "review",
        "block"
    ]
)

flags_valid = isinstance(
    integration_decision["rule_flags"],
    list
)

reason_valid = (
    isinstance(
        integration_decision[
            "human_readable_reason"
        ],
        str
    )
    and
    len(
        integration_decision[
            "human_readable_reason"
        ].strip()
    ) > 0
)

print("========== DECISION OUTPUT VALIDATION ==========")

print(
    "All required fields present:",
    len(missing_decision_fields) == 0
)

print(
    "Risk score valid:",
    risk_score_valid
)

print(
    "ML score valid:",
    ml_score_valid
)

print(
    "Decision valid:",
    decision_valid
)

print(
    "Rule flags valid:",
    flags_valid
)

print(
    "Human-readable reason valid:",
    reason_valid
)

print(
    "Overall output valid:",
    all([
        len(missing_decision_fields) == 0,
        risk_score_valid,
        ml_score_valid,
        decision_valid,
        flags_valid,
        reason_valid
    ])
)

print("=================================================")

========== DECISION OUTPUT VALIDATION ==========
All required fields present: True
Risk score valid: True
ML score valid: True
Decision valid: True
Rule flags valid: True
Human-readable reason valid: True
Overall output valid: True


In [11]:
from xgboost import XGBClassifier

MODEL_PATH = MODELS_DIR / "xgboost_fraud_detector.json"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model file not found:\n{MODEL_PATH}"
    )

model = XGBClassifier()

model.load_model(
    MODEL_PATH
)

sample_df = df.head(
    min(100, len(df))
).copy()

real_results = []

for _, row in sample_df.iterrows():

    transaction = {
        column: row[column]
        for column in ML_FEATURES
    }

    rule_result = evaluate_rules(
        transaction
    )

    ml_probability = float(
        model.predict_proba(
            pd.DataFrame(
                [transaction],
                columns=ML_FEATURES
            )
        )[0, 1]
    )

    decision_result = make_decision(
        ml_fraud_score=ml_probability,
        rule_result=rule_result,
        config=DECISION_CONFIG
    )

    real_results.append(
        decision_result
    )

real_results_df = pd.DataFrame(
    real_results
)

print("========== REAL TRANSACTION TEST ==========")
print(
    "Transactions processed:",
    len(real_results_df)
)

print(
    "Decision counts:"
)

print(
    real_results_df["decision"]
    .value_counts()
)

print(
    "Risk score minimum:",
    real_results_df["risk_score"].min()
)

print(
    "Risk score maximum:",
    real_results_df["risk_score"].max()
)

print("===========================================")

========== REAL TRANSACTION TEST ==========
Transactions processed: 100
Decision counts:
decision
allow    96
otp       4
Name: count, dtype: int64
Risk score minimum: 0.00048019702080637217
Risk score maximum: 0.33982932567596436


In [12]:
all_real_risk_scores_valid = (
    real_results_df["risk_score"]
    .between(0, 1)
    .all()
)

all_real_ml_scores_valid = (
    real_results_df["ml_fraud_score"]
    .between(0, 1)
    .all()
)

all_real_decisions_valid = (
    real_results_df["decision"]
    .isin([
        "allow",
        "otp",
        "review",
        "block"
    ])
    .all()
)

all_real_flags_valid = all(
    isinstance(
        flags,
        list
    )
    for flags
    in real_results_df["rule_flags"]
)

all_real_reasons_valid = all(
    isinstance(
        reason,
        str
    )
    and
    len(
        reason.strip()
    ) > 0
    for reason
    in real_results_df[
        "human_readable_reason"
    ]
)

print("========== REAL DECISION VALIDATION ==========")

print(
    "All risk scores valid:",
    all_real_risk_scores_valid
)

print(
    "All ML scores valid:",
    all_real_ml_scores_valid
)

print(
    "All decisions valid:",
    all_real_decisions_valid
)

print(
    "All rule flags valid:",
    all_real_flags_valid
)

print(
    "All explanations valid:",
    all_real_reasons_valid
)

print(
    "Real transaction validation passed:",
    all([
        all_real_risk_scores_valid,
        all_real_ml_scores_valid,
        all_real_decisions_valid,
        all_real_flags_valid,
        all_real_reasons_valid
    ])
)

print("============================================")

========== REAL DECISION VALIDATION ==========
All risk scores valid: True
All ML scores valid: True
All decisions valid: True
All rule flags valid: True
All explanations valid: True
Real transaction validation passed: True


In [13]:
DECISION_CONFIG_PATH = (
    MODELS_DIR
    / "decision_engine_config.json"
)

with open(
    DECISION_CONFIG_PATH,
    "w"
) as file:
    json.dump(
        DECISION_CONFIG,
        file,
        indent=4
    )

print("========== DECISION CONFIG SAVED ==========")
print(
    "Path:",
    DECISION_CONFIG_PATH
)

print(
    "File exists:",
    DECISION_CONFIG_PATH.exists()
)

print("===========================================")

========== DECISION CONFIG SAVED ==========
Path: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\decision_engine_config.json
File exists: True


In [14]:
DECISION_ENGINE_CODE = '''from pathlib import Path
import json
import numpy as np

SRC_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = SRC_DIR.parent
MODELS_DIR = PROJECT_ROOT / "models"

CONFIG_PATH = MODELS_DIR / "decision_engine_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Decision engine configuration not found: {CONFIG_PATH}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    DECISION_CONFIG = json.load(file)


def calculate_risk_score(
    ml_fraud_score,
    rule_score
):
    ml_fraud_score = float(ml_fraud_score)
    rule_score = float(rule_score)

    if not 0 <= ml_fraud_score <= 1:
        raise ValueError(
            "ML fraud score must be between 0 and 1."
        )

    if not 0 <= rule_score <= 1:
        raise ValueError(
            "Rule score must be between 0 and 1."
        )

    risk_score = (
        DECISION_CONFIG["ml_weight"]
        * ml_fraud_score
        +
        DECISION_CONFIG["rule_weight"]
        * rule_score
    )

    return float(
        np.clip(
            risk_score,
            0.0,
            1.0
        )
    )


def classify_risk(risk_score):
    risk_score = float(risk_score)

    if not 0 <= risk_score <= 1:
        raise ValueError(
            "Risk score must be between 0 and 1."
        )

    if risk_score <= DECISION_CONFIG["allow_max_risk"]:
        return "allow"

    if risk_score <= DECISION_CONFIG["otp_max_risk"]:
        return "otp"

    if risk_score <= DECISION_CONFIG["review_max_risk"]:
        return "review"

    return "block"


def generate_decision_reason(
    ml_fraud_score,
    rule_flags,
    risk_score,
    decision
):
    if not isinstance(
        rule_flags,
        list
    ):
        raise TypeError(
            "rule_flags must be a list."
        )

    if rule_flags:
        rule_text = ", ".join(
            rule_flags
        )
        rule_reason = (
            f"Triggered rules: {rule_text}"
        )
    else:
        rule_reason = (
            "No deterministic rules triggered"
        )

    return (
        f"{rule_reason}. "
        f"ML fraud score: "
        f"{float(ml_fraud_score):.4f}. "
        f"Combined risk score: "
        f"{float(risk_score):.4f}. "
        f"Decision: {decision.upper()}."
    )


def make_decision(
    ml_fraud_score,
    rule_result
):
    if not isinstance(
        rule_result,
        dict
    ):
        raise TypeError(
            "rule_result must be a dictionary."
        )

    if "rule_flags" not in rule_result:
        raise ValueError(
            "rule_result is missing rule_flags."
        )

    if "rule_score" not in rule_result:
        raise ValueError(
            "rule_result is missing rule_score."
        )

    rule_flags = rule_result["rule_flags"]
    rule_score = float(
        rule_result["rule_score"]
    )

    risk_score = calculate_risk_score(
        ml_fraud_score,
        rule_score
    )

    decision = classify_risk(
        risk_score
    )

    human_readable_reason = (
        generate_decision_reason(
            ml_fraud_score,
            rule_flags,
            risk_score,
            decision
        )
    )

    return {
        "risk_score": risk_score,
        "rule_flags": rule_flags,
        "ml_fraud_score": float(
            ml_fraud_score
        ),
        "decision": decision,
        "human_readable_reason": human_readable_reason
    }
'''

DECISION_ENGINE_PATH = (
    SRC_DIR
    / "decision_engine.py"
)

DECISION_ENGINE_PATH.write_text(
    DECISION_ENGINE_CODE,
    encoding="utf-8"
)

print("========== DECISION ENGINE MODULE ==========")
print(
    "File:",
    DECISION_ENGINE_PATH
)

print(
    "File exists:",
    DECISION_ENGINE_PATH.exists()
)

print("============================================")

========== DECISION ENGINE MODULE ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src\decision_engine.py
File exists: True


In [15]:
from decision_engine import (
    make_decision as production_make_decision
)

production_result = production_make_decision(
    0.80,
    integration_rule_result
)

notebook_result = make_decision(
    0.80,
    integration_rule_result,
    DECISION_CONFIG
)

production_matches_notebook = (
    production_result
    == notebook_result
)

print("========== PRODUCTION DECISION TEST ==========")
print(
    "Production result:",
    production_result
)

print(
    "Notebook result:",
    notebook_result
)

print(
    "Production matches notebook:",
    production_matches_notebook
)

print("==============================================")

========== PRODUCTION DECISION TEST ==========
Production result: {'risk_score': 0.9, 'rule_flags': ['HIGH_AMOUNT', 'HIGH_VELOCITY', 'IMPOSSIBLE_TRAVEL', 'NEW_MERCHANT_CATEGORY'], 'ml_fraud_score': 0.8, 'decision': 'block', 'human_readable_reason': 'Triggered rules: HIGH_AMOUNT, HIGH_VELOCITY, IMPOSSIBLE_TRAVEL, NEW_MERCHANT_CATEGORY. ML fraud score: 0.8000. Combined risk score: 0.9000. Decision: BLOCK.'}
Notebook result: {'risk_score': 0.9, 'rule_flags': ['HIGH_AMOUNT', 'HIGH_VELOCITY', 'IMPOSSIBLE_TRAVEL', 'NEW_MERCHANT_CATEGORY'], 'ml_fraud_score': 0.8, 'decision': 'block', 'human_readable_reason': 'Triggered rules: HIGH_AMOUNT, HIGH_VELOCITY, IMPOSSIBLE_TRAVEL, NEW_MERCHANT_CATEGORY. ML fraud score: 0.8000. Combined risk score: 0.9000. Decision: BLOCK.'}
Production matches notebook: True


In [16]:
production_decision_results = []

for _, row in sample_df.iterrows():

    transaction = {
        column: row[column]
        for column in ML_FEATURES
    }

    rule_result = evaluate_rules(
        transaction
    )

    ml_probability = float(
        model.predict_proba(
            pd.DataFrame(
                [transaction],
                columns=ML_FEATURES
            )
        )[0, 1]
    )

    result = production_make_decision(
        ml_probability,
        rule_result
    )

    production_decision_results.append(
        result
    )

production_decision_df = pd.DataFrame(
    production_decision_results
)

production_risk_valid = (
    production_decision_df[
        "risk_score"
    ]
    .between(0, 1)
    .all()
)

production_ml_valid = (
    production_decision_df[
        "ml_fraud_score"
    ]
    .between(0, 1)
    .all()
)

production_decisions_valid = (
    production_decision_df[
        "decision"
    ]
    .isin([
        "allow",
        "otp",
        "review",
        "block"
    ])
    .all()
)

print("========== PRODUCTION DECISION BATCH ==========")

print(
    "Transactions processed:",
    len(production_decision_df)
)

print(
    "Risk scores valid:",
    production_risk_valid
)

print(
    "ML scores valid:",
    production_ml_valid
)

print(
    "Decisions valid:",
    production_decisions_valid
)

print(
    "Batch test passed:",
    all([
        len(production_decision_df)
        == len(sample_df),
        production_risk_valid,
        production_ml_valid,
        production_decisions_valid
    ])
)

print("===============================================")

========== PRODUCTION DECISION BATCH ==========
Transactions processed: 100
Risk scores valid: True
ML scores valid: True
Decisions valid: True
Batch test passed: True


In [17]:
decision_validation = {
    "decision_levels": [
        "allow",
        "otp",
        "review",
        "block"
    ],
    "four_decision_tests_passed": all(
        decision_tests_passed.values()
    ),
    "rule_decision_integration_passed": (
        integration_decision["decision"]
        in [
            "allow",
            "otp",
            "review",
            "block"
        ]
    ),
    "output_validation_passed": all([
        len(missing_decision_fields) == 0,
        risk_score_valid,
        ml_score_valid,
        decision_valid,
        flags_valid,
        reason_valid
    ]),
    "real_transaction_validation_passed": all([
        all_real_risk_scores_valid,
        all_real_ml_scores_valid,
        all_real_decisions_valid,
        all_real_flags_valid,
        all_real_reasons_valid
    ]),
    "production_matches_notebook": (
        production_matches_notebook
    ),
    "production_batch_test_passed": all([
        len(production_decision_df)
        == len(sample_df),
        production_risk_valid,
        production_ml_valid,
        production_decisions_valid
    ])
}

DECISION_VALIDATION_PATH = (
    METRICS_DIR
    / "decision_engine_validation.json"
)

with open(
    DECISION_VALIDATION_PATH,
    "w"
) as file:
    json.dump(
        decision_validation,
        file,
        indent=4
    )

print("========== DECISION VALIDATION SAVED ==========")
print(
    "Path:",
    DECISION_VALIDATION_PATH
)

print(
    "File exists:",
    DECISION_VALIDATION_PATH.exists()
)

print("===============================================")

========== DECISION VALIDATION SAVED ==========
Path: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\decision_engine_validation.json
File exists: True


In [22]:
final_decision_engine_ready = all([
    all(
        decision_tests_passed.values()
    ),
    production_matches_notebook,
    all_real_risk_scores_valid,
    all_real_ml_scores_valid,
    all_real_decisions_valid,
    all_real_flags_valid,
    all_real_reasons_valid,
    production_risk_valid,
    production_ml_valid,
    production_decisions_valid,
    DECISION_CONFIG_PATH.exists(),
    DECISION_ENGINE_PATH.exists(),
    DECISION_VALIDATION_PATH.exists()
])

print()
print("================================================")
print("    STREAMSENTINEL V2 — PHASE 10 SUMMARY")
print("================================================")

print()

print("Decision levels:")
print("Allow")
print("OTP")
print("Review")
print("Block")

print()

print(
    "All four decision tests passed:",
    all(
        decision_tests_passed.values()
    )
)

print(
    "Rule + decision integration passed:",
    integration_decision["decision"]
    in [
        "allow",
        "otp",
        "review",
        "block"
    ]
)

print(
    "Decision output validation passed:",
    all([
        len(missing_decision_fields) == 0,
        risk_score_valid,
        ml_score_valid,
        decision_valid,
        flags_valid,
        reason_valid
    ])
)

print(
    "Real transaction validation passed:",
    all([
        all_real_risk_scores_valid,
        all_real_ml_scores_valid,
        all_real_decisions_valid,
        all_real_flags_valid,
        all_real_reasons_valid
    ])
)

print(
    "Production module matches notebook:",
    production_matches_notebook
)

print(
    "Production batch test passed:",
    all([
        len(production_decision_df)
        == len(sample_df),
        production_risk_valid,
        production_ml_valid,
        production_decisions_valid
    ])
)

print(
    "Decision configuration saved:",
    DECISION_CONFIG_PATH.exists()
)

print(
    "Decision engine module saved:",
    DECISION_ENGINE_PATH.exists()
)

print(
    "Validation results saved:",
    DECISION_VALIDATION_PATH.exists()
)

print()
print(
    "FINAL DECISION ENGINE STATUS:",
    "READY"
    if final_decision_engine_ready
    else "NOT READY"
)

print(
    "Overall verification:",
    final_decision_engine_ready
)

print("================================================")


    STREAMSENTINEL V2 — PHASE 10 SUMMARY

Decision levels:
Allow
OTP
Review
Block

All four decision tests passed: True
Rule + decision integration passed: True
Decision output validation passed: True
Real transaction validation passed: True
Production module matches notebook: True
Production batch test passed: True
Decision configuration saved: True
Decision engine module saved: True
Validation results saved: True

FINAL DECISION ENGINE STATUS: READY
Overall verification: True
